# Exploration vs Exploitation — The Bandit Problem

All previous notebooks assumed we had enough data. Matrix factorization needs ratings. Neural CF needs ratings. Sequential recommendation needs a history of interactions.

But what about a new item with zero ratings? If you never recommend it, you never collect data about it. If you always recommend proven items, you miss potentially great ones and users see the same things forever.

This notebook is about that problem. It is also the entry point to reinforcement learning.

In [ ]:
import random, math, matplotlib.pyplot as plt
%matplotlib inline

## Step 1: The Problem All Recommenders Face

Every recommender system makes a choice at each step: which item to show?

Two goals pull in opposite directions:

- **Exploit** — show items you know the user likes. Safe, but you never learn about new items.
- **Explore** — show items you know little about. You gather information, but the user might not enjoy it.

Get this wrong and the consequences are real. Over-exploit and you create a **filter bubble**: users see only what they already know they like, engagement plateaus, and new items starve. Over-explore and you waste recommendations on items the user dislikes, engagement drops, and users churn.

The right policy explores enough to discover good items and exploits enough to deliver value. That balance is the **exploration/exploitation trade-off**.

### Why this is different from the notebooks above

Notebooks 1–5 were **offline**: we trained on a fixed dataset, then evaluated on held-out data. The model never influenced which data it received.

This notebook is **online**: the system recommends, observes the outcome, updates its beliefs, and recommends again. The data it collects depends on what it chose to recommend. This feedback loop is what makes the problem interesting — and hard.

## Step 2: The Multi-Armed Bandit

Formalise the problem. There are `N` items (the **arms** of a slot machine). Each arm has an unknown true **engagement rate** θᵢ — the probability that a user clicks if shown item `i`. You recommend one item per timestep, observe a click (1) or no click (0), and update your beliefs. Goal: maximise total clicks over `T` timesteps.

This is called the **multi-armed bandit problem**. The name comes from a row of slot machines ('one-armed bandits'). Each machine has an unknown payout rate; you want to find the best one while losing as little money as possible.

**Regret** measures the cost of not knowing which arm is best. At each step, the best possible action would have been to pull the arm with the highest true rate θ*. **Regret** is the gap between that ideal and what you actually earned:

```
regret at step t = θ* - θ_{arm pulled at t}
cumulative regret = sum of regret over all steps
```

A good strategy minimises cumulative regret. It should grow slowly — ideally `O(log T)` — rather than linearly.

In [ ]:
random.seed(42)

N_ARMS = 20
T = 1000  # number of timesteps

# True engagement rates — hidden from the agent, known only to the simulator
TRUE_RATES = [random.betavariate(2, 5) for _ in range(N_ARMS)]

def pull(arm):
    """Simulate a user click: 1 if clicked, 0 if not."""
    return 1 if random.random() < TRUE_RATES[arm] else 0

BEST_ARM = max(range(N_ARMS), key=lambda a: TRUE_RATES[a])
BEST_RATE = TRUE_RATES[BEST_ARM]

print(f'Best arm: {BEST_ARM} (true rate: {BEST_RATE:.3f})')
print(f'\nTrue rates (sorted):')
sorted_rates = sorted(enumerate(TRUE_RATES), key=lambda x: -x[1])
for arm, rate in sorted_rates[:5]:
    print(f'  arm {arm:2d}: {rate:.3f}')
print(f'  ...')
print(f'  (worst): {sorted_rates[-1][1]:.3f}')

In [ ]:
def compute_cumulative_regret(chosen_arms):
    """Given a sequence of chosen arms, compute cumulative regret at each step."""
    regrets = []
    total = 0.0
    for arm in chosen_arms:
        total += BEST_RATE - TRUE_RATES[arm]
        regrets.append(total)
    return regrets

# Pure greedy baseline (always pull arm 0, ignoring all information)
always_zero = [0] * T
print(f'Always pull arm 0 — cumulative regret after {T} steps: '
      f'{compute_cumulative_regret(always_zero)[-1]:.1f}')
print(f'Oracle (always pull best arm) — cumulative regret: 0.0')

## Step 3: ε-Greedy

The simplest strategy with exploration baked in:

- With probability **ε**: choose a random arm (explore)
- With probability **1 − ε**: choose the arm with the highest estimated rate (exploit)

Estimate each arm's rate as the fraction of pulls that resulted in a click: `successes / n_pulls`. Arms that have never been pulled get an estimated rate of 0, so the strategy forces exploration at the start.

The parameter ε controls the trade-off:
- **ε = 0** (pure exploit): get stuck on the first arm that looks good. If that arm is suboptimal, you never recover.
- **ε = 1** (pure explore): random recommendations forever. Finds the best arm eventually but never exploits it.
- **ε = 0.1**: a common default. Explore 10% of the time, exploit 90%.

In [ ]:
def run_epsilon_greedy(epsilon, seed=0):
    random.seed(seed)
    counts   = [0] * N_ARMS   # how many times each arm was pulled
    successes = [0] * N_ARMS  # how many clicks each arm received
    chosen_arms = []

    for t in range(T):
        if random.random() < epsilon:
            # Explore: random arm
            arm = random.randrange(N_ARMS)
        else:
            # Exploit: arm with highest estimated rate
            # (0 / 0 = 0, so unexplored arms start at 0)
            rates = [successes[a] / counts[a] if counts[a] > 0 else 0.0
                     for a in range(N_ARMS)]
            arm = max(range(N_ARMS), key=lambda a: rates[a])

        reward = pull(arm)
        counts[arm] += 1
        successes[arm] += reward
        chosen_arms.append(arm)

    return compute_cumulative_regret(chosen_arms)


# Compare three epsilon values
epsilons = [0.0, 0.1, 0.3]
colors = ['tab:red', 'tab:blue', 'tab:orange']

plt.figure(figsize=(8, 4))
for eps, color in zip(epsilons, colors):
    regrets = run_epsilon_greedy(eps)
    label = f'e={eps} (pure exploit)' if eps == 0 else f'e={eps}'
    plt.plot(regrets, color=color, label=label)
plt.xlabel('Timestep'); plt.ylabel('Cumulative regret')
plt.title('e-Greedy: Effect of epsilon')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 4: Upper Confidence Bound (UCB)

ε-greedy explores randomly. When it explores, it's equally likely to pull the best unknown arm and the worst unknown arm. That's wasteful.

A better idea: explore arms you are **uncertain about**. If you have pulled arm 3 only once, your estimate of its rate is unreliable — it could be much higher or lower than the observed value. If you have pulled arm 7 a hundred times, your estimate is tight. Prefer arms where uncertainty leaves open the possibility of being the best.

**UCB (Upper Confidence Bound)** makes this precise. At each step, score each arm as:

```
UCB_score(i) = estimated_rate(i) + sqrt(2 * log(t) / n_i)
```

- `estimated_rate(i)` = successes / pulls. The exploitation term.
- `sqrt(2 * log(t) / n_i)` = the optimism bonus. Large when `n_i` is small (uncertain arm). Shrinks as we pull it more. The exploration term.

Always recommend the arm with the highest UCB score. Arms you haven't explored much get a large bonus, making them attractive. Once you've pulled them enough, their bonus shrinks and you exploit if they're good, ignore them if they're not.

This approach is called **optimism in the face of uncertainty**: treat uncertain arms as if they *might* be the best, and prove otherwise by trying them.

In [ ]:
def run_ucb(seed=0):
    random.seed(seed)
    counts    = [0] * N_ARMS
    successes = [0] * N_ARMS
    chosen_arms = []

    # Pull each arm once to initialise estimates
    for arm in range(N_ARMS):
        reward = pull(arm)
        counts[arm] += 1
        successes[arm] += reward
        chosen_arms.append(arm)

    for t in range(N_ARMS, T):
        # UCB score: estimated rate + optimism bonus
        scores = [
            successes[a] / counts[a] + math.sqrt(2 * math.log(t + 1) / counts[a])
            for a in range(N_ARMS)
        ]
        arm = max(range(N_ARMS), key=lambda a: scores[a])
        reward = pull(arm)
        counts[arm] += 1
        successes[arm] += reward
        chosen_arms.append(arm)

    return compute_cumulative_regret(chosen_arms)


plt.figure(figsize=(8, 4))
plt.plot(run_epsilon_greedy(0.1), color='tab:blue',   label='e-Greedy (e=0.1)')
plt.plot(run_ucb(),               color='tab:green',  label='UCB')
plt.xlabel('Timestep'); plt.ylabel('Cumulative regret')
plt.title('e-Greedy vs UCB')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 5: Thompson Sampling

UCB commits to a single point estimate plus a fixed confidence interval. Thompson Sampling takes a different approach: maintain a full **probability distribution** over each arm's true rate, and let randomness drive exploration.

### The Beta distribution

We model each arm's true rate θᵢ with a **Beta distribution** — the natural prior for an unknown probability. `Beta(α, β)` is defined on `[0, 1]`. The parameters track what we've seen:

- **α** = 1 + number of successes (clicks)
- **β** = 1 + number of failures (no clicks)

Start with `Beta(1, 1)` — the uniform distribution, total ignorance. After 3 clicks and 7 non-clicks: `Beta(4, 8)`, a distribution centred around 0.3.

### The algorithm

At each step:
1. For each arm, **sample** θᵢ ~ `Beta(αᵢ, βᵢ)`
2. Recommend the arm with the highest sampled θᵢ
3. Observe the reward, update α or β accordingly

If an arm has wide uncertainty, its samples vary a lot — sometimes high, sometimes low. On a high sample, it gets recommended and we learn more about it. If it's actually bad, the samples will stop being high and the arm gets dropped. This is Bayesian reasoning doing exploration automatically, without any ε or confidence formula.

In [ ]:
def beta_sample(alpha, beta_param):
    """Sample from Beta(alpha, beta) using the Gamma distribution relationship.

    If X ~ Gamma(alpha, 1) and Y ~ Gamma(beta, 1), then X/(X+Y) ~ Beta(alpha, beta).
    Python's random.gammavariate(a, b) samples from Gamma(shape=a, scale=b).
    """
    x = random.gammavariate(alpha, 1)
    y = random.gammavariate(beta_param, 1)
    return x / (x + y)

# Sanity check: mean of Beta(alpha, beta) = alpha / (alpha + beta)
random.seed(0)
samples = [beta_sample(4, 8) for _ in range(10000)]
empirical_mean = sum(samples) / len(samples)
theoretical_mean = 4 / (4 + 8)
print(f'Beta(4, 8): empirical mean={empirical_mean:.4f}, theoretical={theoretical_mean:.4f}')

In [ ]:
def run_thompson(seed=0):
    random.seed(seed)
    # Start with Beta(1,1) = uniform prior on each arm's true rate
    alpha = [1] * N_ARMS  # 1 + successes
    beta  = [1] * N_ARMS  # 1 + failures
    chosen_arms = []

    for t in range(T):
        # Sample a rate estimate for each arm from its current Beta distribution
        sampled = [beta_sample(alpha[a], beta[a]) for a in range(N_ARMS)]
        arm = max(range(N_ARMS), key=lambda a: sampled[a])

        reward = pull(arm)
        if reward == 1:
            alpha[arm] += 1
        else:
            beta[arm] += 1
        chosen_arms.append(arm)

    return compute_cumulative_regret(chosen_arms)


# Compare all three strategies
plt.figure(figsize=(8, 4))
plt.plot(run_epsilon_greedy(0.1), color='tab:blue',   label='e-Greedy (e=0.1)')
plt.plot(run_ucb(),               color='tab:green',  label='UCB')
plt.plot(run_thompson(),          color='tab:purple', label='Thompson Sampling')
plt.xlabel('Timestep'); plt.ylabel('Cumulative regret')
plt.title('Bandit Strategies: Cumulative Regret')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Step 6: The Connection to Reinforcement Learning

Pull back and look at the structure of everything we just built:

- A **policy**: which arm to choose at each step (ε-greedy, UCB, or Thompson Sampling)
- An **environment**: the true engagement rates, which the policy can't see
- A **reward signal**: the click or no-click observed after each recommendation
- An **update rule**: adjust beliefs based on the reward

This is reinforcement learning. The agent (policy) interacts with the environment, observes rewards, and improves its behaviour.

The multi-armed bandit is RL with one simplification: **actions don't change the state**. Recommending item 3 this step doesn't make item 4 harder or easier to click next step. Each step is independent.

In full RL, actions change the state. The next state depends on what you did. In chess, a move changes the board. In robotics, a motor command changes the robot's position. In conversation, the last message changes the context. That state transition is the only thing missing from bandit problems.

| Bandit | Full RL |
|--------|---------|
| Arms | Actions |
| True engagement rates | Environment dynamics |
| Click / no-click | Reward |
| No state — each step is fresh | State transitions |
| Thompson Sampling, UCB | Policy gradient, PPO, Q-learning |

### RLHF: where this shows up in LLMs

The RLHF pipeline that fine-tunes language models is full RL:
- **Policy**: the language model
- **Action**: the next token generated
- **State**: the conversation so far
- **Reward**: a human preference score (or a reward model trained on human preferences)

The exploration/exploitation problem appears at every scale, from showing a user a new movie to deciding which word to generate next. The bandit is the simplest setting where you can see the structure clearly.

In [ ]:
# Visualise how Thompson Sampling's beliefs evolve over time.
# Show the Beta distributions for the best and worst arms at t=10, t=100, t=1000.

def simulate_thompson_beliefs(n_steps, seed=0):
    random.seed(seed)
    alpha = [1] * N_ARMS
    beta  = [1] * N_ARMS
    for t in range(n_steps):
        sampled = [beta_sample(alpha[a], beta[a]) for a in range(N_ARMS)]
        arm = max(range(N_ARMS), key=lambda a: sampled[a])
        reward = pull(arm)
        if reward == 1:
            alpha[arm] += 1
        else:
            beta[arm] += 1
    return alpha, beta

def beta_pdf(x, a, b):
    """Unnormalised Beta(a,b) density: x^(a-1) * (1-x)^(b-1)."""
    if x <= 0 or x >= 1:
        return 0
    return x**(a-1) * (1-x)**(b-1)

xs = [i/200 for i in range(1, 200)]

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n_steps in zip(axes, [10, 100, 1000]):
    a, b = simulate_thompson_beliefs(n_steps)
    for arm, color, label in [(BEST_ARM, 'tab:green', 'best arm'),
                               (sorted_rates[-1][0], 'tab:red', 'worst arm')]:
        ys = [beta_pdf(x, a[arm], b[arm]) for x in xs]
        max_y = max(ys) or 1
        ys = [y / max_y for y in ys]  # normalise for plotting
        ax.plot(xs, ys, color=color, label=label)
    ax.axvline(TRUE_RATES[BEST_ARM], color='tab:green', linestyle='--', alpha=0.5)
    ax.set_title(f't={n_steps}')
    ax.set_xlabel('Engagement rate')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.suptitle('Thompson Sampling: Beliefs at different timesteps (normalised)')
plt.tight_layout()
plt.show()

sorted_rates = sorted(enumerate(TRUE_RATES), key=lambda x: -x[1])
print(f'Best arm true rate:  {TRUE_RATES[BEST_ARM]:.3f}')
print(f'Worst arm true rate: {TRUE_RATES[sorted_rates[-1][0]]:.3f}')

## Summary

| Strategy | Exploration mechanism | Convergence | Limitation |
|----------|-----------------------|-------------|------------|
| Greedy (ε=0) | None | Never escapes suboptimal arm | Stuck on first good arm found |
| ε-Greedy | Random with fixed probability | Slow, keeps exploring forever | Wastes exploration on known-bad arms |
| UCB | Optimism bonus proportional to uncertainty | `O(log T)` regret | Tuning-free but can be slow in early steps |
| Thompson Sampling | Sample from posterior distribution | Fast, often best in practice | Requires conjugate prior or approximate inference |

### What we learned

- **Exploration and exploitation conflict.** You can't always do both — every recommendation is either learning or earning.
- **Regret quantifies the cost of uncertainty.** A good strategy keeps cumulative regret growing slowly.
- **ε-Greedy is the baseline.** Simple, interpretable, works. But wastes exploration on arms already known to be bad.
- **UCB is principled exploration.** Explore in proportion to uncertainty, not at random.
- **Thompson Sampling reasons from distributions.** Exploration emerges from the width of the posterior — no tuning required.
- **Bandits are RL without state transitions.** The structure — policy, environment, reward, update — is identical to full RL.

---

This notebook closes Course 2. The arc: matrix completion — collaborative filtering — latent factors — neural models — sequential patterns — learning from feedback. The same four-step training loop from ML Foundations runs through all of them.

> **Course 3** — coming soon.

## Your turn

### 1. Watch uncertainty collapse

The `simulate_thompson_beliefs` function above tracks beliefs up to `n_steps`. Extend it to plot the Beta distributions for the best and worst arms at `t=10`, `t=100`, and `t=1000` in a single row of subplots. Describe what you observe: how does the width of each distribution change? At `t=1000`, does the best arm's distribution sit near its true rate?

### 2. Decaying epsilon

Modify `run_epsilon_greedy` to accept a `decay` parameter. Instead of a fixed `epsilon`, use `epsilon * (1 - decay) ** t` at step `t`. Start with `epsilon=0.3` and try `decay=0.003` (which brings epsilon to ~0.05 by step 1000).

Plot cumulative regret for: fixed ε=0.1, fixed ε=0.3, and decaying ε. Does scheduled decay beat fixed ε? At what point in training does the switch from exploration to exploitation happen, and does earlier or later switching help?

### 3. LinUCB — the contextual bandit

In this notebook, each arm's rate was fixed. In a real recommender, the rate depends on the user: action films are great for user A and terrible for user B.

Extend the simulation: assign each arm a feature vector `v_i` of length 4 (random, fixed at the start). Assign each user a preference vector `w` of length 4. The true click probability for user `u` and arm `i` is `sigmoid(w · v_i)`. Now the best arm depends on the user.

Modify UCB to use a **linear model** for estimating click rates: fit a ridge regression on observed (features, reward) pairs for each arm. Use the fitted model's prediction as the estimated rate, and the regression uncertainty as the confidence bonus. This is **LinUCB**, the foundation of modern contextual recommendation systems.